In [1]:
!pip install streamlit

In [2]:
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
import streamlit as st

In [3]:
df=pd.read_csv('/kaggle/input/datasets/ddsstt/final-recommendation/df_perdict_cluster.csv')

In [4]:
cluster_model_path= '/kaggle/input/models/ddsstt/cluster-model-jo/scikitlearn/default/1/cluster.joblib'

In [5]:
df.head()

,name,online_order,book_table,rate,votes,location,rest_type,cost2pepole,type,item_id,...,votes_per_chain,is_standalone,full_service,menu_diversity,is_specialized,success_map,location_simplified,location_freq,type_freq,cluster
0,Jalsa,Yes,Yes,4.1,775,banashankari,casual dining,800.0,buffet,0,...,64.583333,0,1,3,0,3,banashankari,0.017707,0.017023,0
1,Spice Elephant,Yes,No,4.1,787,banashankari,casual dining,800.0,buffet,1,...,157.400000,0,0,3,0,3,banashankari,0.017707,0.017023,1
2,San Churro Cafe,Yes,No,3.8,918,banashankari,cafe& casual dining,800.0,buffet,2,...,76.500000,0,0,6,0,2,banashankari,0.017707,0.017023,0
3,Addhuri Udupi Bhojana,No,No,3.7,88,banashankari,quick bites,300.0,buffet,3,...,29.333333,0,0,2,1,2,banashankari,0.017707,0.017023,1
4,Grand Village,No,No,3.8,166,basavanagudi,casual dining,600.0,buffet,4,...,33.200000,0,0,2,1,2,basavanagudi,0.013368,0.017023,1


In [6]:
df=df.dropna(subset=['primary_cuisine'])

In [7]:
df['online_order'] = df['online_order'].map({'Yes': 1, 'No': 0})
df['book_table'] = df['book_table'].map({'Yes': 1, 'No': 0})
df = df.drop(columns=['online_map', 'book_map'])

In [8]:
df= df.drop_duplicates(
    subset=['name', 'online_order', 'book_table', 'rate', 'votes', 'location',
       'rest_type', 'cost2pepole', 'type', 'rate_missing',
       'chain_size', 'num_of_rest_types', 'cuisines_count', 'cost_bucket',
       'cuisines_list', 'log_votes', 'log_cost', 'success', 'primary_cuisine',
       'rate_votes_interaction', 'rate_per_cost', 'cost_per_cuisine',
       'votes_per_chain', 'is_standalone', 'full_service', 'menu_diversity',
       'is_specialized', 'success_map', 'location_simplified', 'location_freq',
       'type_freq', 'cluster'],
    keep='first'
)

In [9]:
df.shape

(31574, 33)

# layer 1 (scenario)

In [10]:
class ScenarioEngine:
    
    def __init__(self):
        self.weights = {
            "date": {
                "book_table": 0.4,
                "budget": 0.4,
                "menu_diversity": 0.2
            },
            "quick_bite": {
                "budget_inverse": 0.4,
                "book_table_inverse": 0.5,
                "menu_diversity_inverse": 0.3
            },
            "casual": {
                "budget_middle": 0.4,
                "flex_booking": 0.3,
                "menu_diversity": 0.3
            }
        }
    
    def _normalize_scores(self, scores):
        total = sum(scores.values())
        if total == 0:
            return scores
        return {k: v/total for k, v in scores.items()}
    
    def detect(self, user_context):
        """
        user_context:
        {
            "budget_norm": float (0–1),
            "book_table": 0/1,
            "menu_diversity_norm": float (0–1)
        }
        """
        
        budget = user_context["budget_norm"]
        book_table = user_context["book_table"]
        menu_div = user_context["menu_diversity_norm"]
        
        scores = {}

        
        # DATE
        scores["date"] = (
            self.weights["date"]["book_table"] * book_table +
            self.weights["date"]["budget"] * budget +
            self.weights["date"]["menu_diversity"] * menu_div
        )
        
        # QUICK BITE
        scores["quick_bite"] = (
            self.weights["quick_bite"]["budget_inverse"] * (1 - budget) +
            self.weights["quick_bite"]["book_table_inverse"] * (1 - book_table) +
            self.weights["quick_bite"]["menu_diversity_inverse"] * (1 - menu_div)
        )
        
        # CASUAL
        # middle_budget_score = 1 - abs(budget - 0.5)
        middle_budget_score = max(0, 1 - abs(budget - 0.5) * 2)
        flex_booking_score = 1 - abs(book_table - 0.5)

        
        scores["casual"] = (
            self.weights["casual"]["budget_middle"] * middle_budget_score +
            self.weights["casual"]["flex_booking"] * flex_booking_score+  
            self.weights["casual"]["menu_diversity"] * menu_div
        )

        
        normalized_scores = self._normalize_scores(scores)
        dominant_scenario = max(normalized_scores, key=normalized_scores.get)
        
        return {
            "scores": normalized_scores,
            "dominant_scenario": dominant_scenario
        }


## ex

In [11]:
# Example user context
user_context = {
    "budget_norm": 0.7,
    "book_table": 1,
    "menu_diversity_norm": 0.6
}

scenario_engine = ScenarioEngine()
scenario_result = scenario_engine.detect(user_context)
print("Scenario Layer:", scenario_result)

Scenario Layer: {'scores': {'date': 0.49689440993788814, 'quick_bite': 0.14906832298136646, 'casual': 0.35403726708074534}, 'dominant_scenario': 'date'}


# layer 2 (similarity)

In [12]:
class CandidateGenerator:
    
    def __init__(self, df, cluster_model_path=None):
        self.df= df.copy()
        self.numeric_features= ['log_cost', 'menu_diversity']
        self.scaler= MinMaxScaler()
        
        # Preprocess numeric
        self.df[self.numeric_features]=self.scaler.fit_transform(self.df[self.numeric_features])
       
        #Clean rest_type using regex
        self.df['rest_type'] =(
            self.df['rest_type']
            .astype(str)
            .str.replace(r'\s*&\s*', ' & ', regex=True)
        )

        # Multi-label rest_type
        self.all_types = sorted(
            {t.strip() for x in self.df['rest_type'] for t in x.split(' &')} )
        
        dummies = self.df['rest_type'].str.get_dummies(sep=' & ')
        dummies = dummies.reindex(columns=self.all_types, fill_value=0)
        self.rest_type_matrix = dummies.values


        #Other features
        self.df['is_specialized']=self.df['is_specialized'].astype(float)
        self.df['cluster']=self.df['cluster'].astype(int)


       # Load cluster model for auto-clustering if path provided
        if cluster_model_path is not None:
            self.kmeans = joblib.load(cluster_model_path)
        else:
            self.kmeans = None  # fallback if no model provided

        #Vectorized Jaccard similarity
    def _vectorized_jaccard(self, x, user_vector):
        """
        x: numpy array of shape (num_rows, num_types)
        user_vector: numpy array of shape (num_types,)
        returns: numpy array of Jaccard similarity for each row
        """
        intersection = np.logical_and(x, user_vector).sum(axis=1)
        union = np.logical_or(x, user_vector).sum(axis=1)
        # return np.divide(intersection, union, out=np.zeros_like(intersection, dtype=float), where=union != 0)
        return np.divide( intersection + 1, union + 1 ) 


    #Candidate generation
    def get_candidates(self, user_pref, top_k=10,weights=None):
        if weights is None:
            weights = {
                'numeric': 0.45,
                'rest_type': 0.30,
                'specialized': 0.10,
                'cluster': 0.15
            }


        # Numeric similarity
        user_num_df = pd.DataFrame([[user_pref[f] for f in self.numeric_features]],
        columns=self.numeric_features)
        user_num = self.scaler.transform(user_num_df)

        numeric_sim = cosine_similarity(self.df[self.numeric_features], user_num).flatten()
        
        # Rest_type similarity
        user_type_multi = np.array(
            [1 if t in user_pref.get('rest_type', []) else 0 for t in self.all_types]
        )
        rest_type_sim = self._vectorized_jaccard(self.rest_type_matrix,user_type_multi)

        #Specialized similarity
        specialized_sim=(
            self.df['is_specialized']== user_pref.get('is_specialized', 0)
        ).astype(float)

        # Cluster similarity (auto)
        if hasattr(self, 'kmeans') and self.kmeans is not None:
            # Prepare user features for clustering (must match kmeans input)
            cluster_features = [
                'online_order',
                'book_table',
                'log_cost',
                'num_of_rest_types',
                'cuisines_count',
                'location_freq',
                'type_freq']
            
            user_cluster_vector = np.array([user_pref.get(f, 0) for f in cluster_features]).reshape(1, -1)
            user_cluster = self.kmeans.predict(user_cluster_vector)[0]
            # cluster_sim = (self.df['cluster'] == user_cluster).astype(float)
            cluster_sim = np.where( self.df['cluster'] == user_cluster, 1.0, 0.3)

        else:
            # fallback if no model, assume all zeros
            cluster_sim = np.zeros(len(self.df))

        #Hybrid similarity
        hybrid_sim = (
            weights['numeric'] * numeric_sim +
            weights['rest_type'] * rest_type_sim +
            weights['specialized'] * specialized_sim +
            weights['cluster'] * cluster_sim
        )

        #Top-K candidates
        top_indices=np.argsort(hybrid_sim)[-top_k:][ ::-1]
        return self.df.iloc[top_indices].copy()

## ex

In [13]:
user_pref = {
    'log_cost': np.log(700),
    'menu_diversity': 3,
    'rest_type': ['casual dining', 'buffet'],
    'is_specialized': 1
}

generator = CandidateGenerator(df)
candidates = generator.get_candidates(user_pref, top_k=5)
print("Candidates Layer 2:")
print(candidates[['name','rest_type', 'cost2pepole', 'menu_diversity', 'is_specialized', 'cluster']])

Candidates Layer 2:
                   name      rest_type  cost2pepole  menu_diversity  \
18893            Haveli  casual dining        300.0        0.066667   
1767    New Kabab Plaza  casual dining        300.0        0.066667   
9060        Swaadam Veg  casual dining        300.0        0.066667   
10396       Swaadam Veg  casual dining        300.0        0.066667   
26873  Krishna Vaibhava  casual dining        350.0        0.066667   

       is_specialized  cluster  
18893             1.0        1  
1767              1.0        1  
9060              1.0        1  
10396             1.0        1  
26873             1.0        1  


# layer 3 (hard filter)

In [14]:
class HardFilter:
    def __init__ (self , df):
        self.df=df.copy()

        # Safety: fill NaN in key columns
        self.df['book_table'] = self.df['book_table'].fillna(0).astype(int)
        self.df['online_order'] = self.df['online_order'].fillna(0).astype(int)

        # Ensure location_simplified is clean
        self.df['location_simplified'] =(
            self.df['location_simplified'].fillna('').str.lower().str.strip())

        # Build lookup: raw location → simplified location
        if 'location' in self.df.columns:
            self.location_map=(
                self.df[['location', 'location_simplified']]
                .dropna()
                .assign(location = lambda x:x ['location'].str.lower().str.strip())
                .drop_duplicates()
                .set_index('location')['location_simplified'].to_dict()
            )
        
        else:
            self.location_map ={}

    def filter_candidates(self, candidates, user_context, scenario=None, relax_order=True):
        """
        candidates: DataFrame from CandidateGenerator
        user_context example:
        {
            'location': 'banashankari',
            'budget_min': 500,
            'budget_max': 1200,
            'book_table': 1,
            'online_order': 1
        }
        scenario: optional string from ScenarioEngine ("date", "quick_bite", "casual")
        relax_order: bool, fallback if no results
        """
        mask = np.ones(len(candidates), dtype=bool)

        #Location filter
        if 'location' in user_context and user_context['location']:
            user_loc_raw= str(user_context['location']).lower().strip()

            # Map to simplified location (fallback → 'other')
            user_loc_simplified= self.location_map.get(user_loc_raw, 'other')
            mask &= (candidates['location_simplified'] == user_loc_simplified)


        #Budget filter
        if 'budget_min' in user_context and 'budget_max' in user_context:
            mask &= (
                (candidates['cost2pepole'] >= user_context['budget_min']) &
                (candidates['cost2pepole'] <= user_context['budget_max'])
            )

        #Booking requirement
        if 'book_table' in user_context:
            if scenario == 'date':
                mask &= (candidates['book_table'] == 1)
            elif scenario == 'quick_bite':
                pass  # ignore booking
            else:
                mask &= (candidates['book_table'] >= user_context['book_table'])


        #Online order requirement
        if 'online_order' in user_context:
            mask &= (candidates['online_order'] == user_context['online_order'])

        filtered_candidates= candidates[mask].copy()

        #Fallback: relax online_order if empty
        if filtered_candidates.empty and relax_order and 'online_order' in user_context:
            relaxed_mask=( mask|(candidates['online_order'] != user_context['online_order'])) 
            filtered_candidates= candidates[relaxed_mask].copy()
            
        return filtered_candidates

## ex

In [15]:
user_context_layer3 = {
    'location': 'banashankari',
    'budget_min': 500,
    'budget_max': 600,
    'book_table': 1,
    'online_order': 0
}

filter_engine = HardFilter(df)
filtered_candidates = filter_engine.filter_candidates(candidates, user_context_layer3)
print("Filtered Candidates Layer 3:")
print(filtered_candidates[['name','rest_type','cost2pepole','book_table','online_order','location_simplified']])


Filtered Candidates Layer 3:
                   name      rest_type  cost2pepole  book_table  online_order  \
9060        Swaadam Veg  casual dining        300.0           0             1   
10396       Swaadam Veg  casual dining        300.0           0             1   
26873  Krishna Vaibhava  casual dining        350.0           0             1   

      location_simplified  
9060                  btm  
10396                 btm  
26873                 btm  


In [16]:
print(filtered_candidates[['name','rest_type', 'cost2pepole', 'book_table', 'online_order', 'location_simplified','menu_diversity', 'is_specialized', 'cluster']])


                   name      rest_type  cost2pepole  book_table  online_order  \
9060        Swaadam Veg  casual dining        300.0           0             1   
10396       Swaadam Veg  casual dining        300.0           0             1   
26873  Krishna Vaibhava  casual dining        350.0           0             1   

      location_simplified  menu_diversity  is_specialized  cluster  
9060                  btm        0.066667             1.0        1  
10396                 btm        0.066667             1.0        1  
26873                 btm        0.066667             1.0        1  


# layer 4 (Ranking)

In [17]:
class Ranker:
    def __init__(self, df):
        self.df = df.copy()

    def rank(self, pref_score, user_pref=None):
        df = self.df.copy()
        df['pref_score'] = pref_score

        # Normalize rating
        rating_range = df['rate'].max() - df['rate'].min()
        df['rating_score'] = (df['rate'] - df['rate'].min()) / max(rating_range, 1e-6)

        # Popularity
        df['popularity_score'] = np.log1p(df['votes'])
        df['popularity_score'] /= df['popularity_score'].max()

        # Business boost
        df['business_boost'] = 1.0
        df.loc[df['is_specialized'] == 1, 'business_boost'] += 0.05

        # Final score
        df['final_score'] = (
            0.45 * df['pref_score'] +
            0.30 * df['rating_score'] +
            0.20 * df['popularity_score'] +
            0.05 * df['business_boost']
        )

        return df.sort_values('final_score', ascending=False)


## ex

In [18]:
# Compute preference similarity from Layer 2 (for ranking)
pref_score = np.array([0.8, 0.75, 0.7, 0.65, 0.6])  # Example similarity scores
# Make sure length matches filtered_candidates
pref_score = pref_score[:len(filtered_candidates)]

# Layer 4: Ranking
ranker = Ranker(filtered_candidates)
final_ranked = ranker.rank(pref_score)

print("\nFinal Ranked Candidates (Layer 4 output):")
print(final_ranked[['name','rest_type', 'pref_score', 'rate', 'votes', 'is_specialized', 'final_score']])


Final Ranked Candidates (Layer 4 output):
                   name      rest_type  pref_score  rate  votes  \
9060        Swaadam Veg  casual dining        0.80   3.6     34   
10396       Swaadam Veg  casual dining        0.75   3.6     34   
26873  Krishna Vaibhava  casual dining        0.70   3.5     41   

       is_specialized  final_score  
9060              1.0     0.902744  
10396             1.0     0.880244  
26873             1.0     0.567500  


# layer5 (re-rank)

In [19]:
class AdaptiveDiversityReranker:
    def __init__(self, 
                 base_cluster_penalty=0.10,
                 base_hidden_gem_bonus=0.10,
                 base_success_bonus=0.10):
        self.base_cluster_penalty = base_cluster_penalty
        self.base_hidden_gem_bonus = base_hidden_gem_bonus
        self.base_success_bonus = base_success_bonus

    def rerank(self, ranked_df, user_level='new'):
        df = ranked_df.copy()
        seen_clusters = {}
        diversified_scores = []


        for _, row in df.iterrows():
            score = row['final_score']
            cluster = row['cluster']

            # 1. Adaptive cluster penalty
            cluster_votes_median = df[df['cluster']==cluster]['log_votes'].median()
            cluster_penalty = self.base_cluster_penalty * (1 - min(cluster_votes_median/10, 0.8))  # scale to ~0-0.8
            count = seen_clusters.get(cluster, 0)
            if count > 0:
                score *= (1 - cluster_penalty * np.log1p(count))
            seen_clusters[cluster] = count + 1

            # 2. Adaptive hidden gem bonus
            hidden_bonus = self.base_hidden_gem_bonus
            if user_level == 'new':
                hidden_bonus *= 1.5  # exploration priority
                
            if row['log_votes'] < cluster_votes_median and row['rate'] >= 3.5:
                score *= (1 + hidden_bonus)

            # 3. Success prior
            if 'success_map' in df.columns:
                success = min(max(row['success_map'],0),1)
                score *= (1 + self.base_success_bonus * success)

            diversified_scores.append(score)

        df['diversified_score'] = diversified_scores
        return df.sort_values('diversified_score', ascending=False)

In [20]:
# Layer 5: Diversity 
diversity_engine = AdaptiveDiversityReranker(
    base_cluster_penalty=0.05,
    base_hidden_gem_bonus=0.10,
    base_success_bonus=0.10
)

final_recommendations = diversity_engine.rerank(final_ranked)

print("Final Recommendations (Layer 5 output):")
print(
    final_recommendations[
        ['name','rest_type', 'cluster', 'votes', 'success_map',
         'final_score', 'diversified_score']
    ]
)

Final Recommendations (Layer 5 output):
                   name      rest_type  cluster  votes  success_map  \
9060        Swaadam Veg  casual dining        1     34            2   
10396       Swaadam Veg  casual dining        1     34            2   
26873  Krishna Vaibhava  casual dining        1     41            2   

       final_score  diversified_score  
9060      0.902744           0.993019  
10396     0.880244           0.946642  
26873     0.567500           0.602151  


# RecommendationPipeline

In [21]:
class RecommendationPipeline:
    def __init__(
        self,
        df,
        scenario_engine,
        candidate_generator,
        filter_engine,
        ranker_class,
        diversity_engine,
        top_k_candidates=50
    ):
        self.df = df
        self.scenario_engine = scenario_engine
        self.candidate_generator = candidate_generator
        self.filter_engine = filter_engine
        self.ranker_class = ranker_class
        self.diversity_engine = diversity_engine
        self.top_k_candidates = top_k_candidates

    def recommend(self, user_context, user_pref, debug=False):
        logs = {}

        # Layer 1: Scenario Detection
        scenario_output = self.scenario_engine.detect(user_context)
        dominant_scenario = scenario_output["dominant_scenario"]
        logs["scenario"] = scenario_output

        # Layer 2: Candidate Generation
        candidates = self.candidate_generator.get_candidates(
            user_pref,
            top_k=self.top_k_candidates
        )
        logs["candidates_count"] = len(candidates)

        # Layer 3: Hard Filtering
        filtered_candidates = self.filter_engine.filter_candidates(
            candidates,
            user_context
        )
        logs["filtered_count"] = len(filtered_candidates)

        if filtered_candidates.empty:
            return {
                "recommendations": filtered_candidates,
                "logs": logs,
                "scenario": dominant_scenario
            }

        # Layer 4: Ranking
        if "pref_score" not in filtered_candidates.columns:
            # recompute using CandidateGenerator similarity
            pref_score = self.candidate_generator.get_candidates(
                user_pref,
                top_k=len(filtered_candidates)
            )
            # Keep only the filtered indices
            pref_score = pref_score.index.map(
                lambda idx: idx in filtered_candidates.index
            ).astype(float)
        else:
            pref_score = filtered_candidates["pref_score"].values

        ranker = self.ranker_class(filtered_candidates)
        ranked = ranker.rank(pref_score)
        logs["avg_final_score"] = ranked["final_score"].mean()

        # Layer 5: Diversity & Exploration
        final_results = self.diversity_engine.rerank(ranked)
        logs["unique_clusters"] = final_results["cluster"].nunique()

        # Output
        if debug:
            return {
                "recommendations": final_results,
                "logs": logs,
                "scenario": dominant_scenario
            }

        return final_results

# main

In [22]:
# from scenario_engine import ScenarioEngine
# from candidate_generator import CandidateGenerator
# from hard_filter import HardFilter
# from ranker import Ranker
# from diversity_reranker import DiversityReranker
# from pipeline import RecommendationPipeline

# Main Entry Point
# =========================
if __name__ == "__main__":

    df = df  
    # Initialize Layers
    print("Initializing engines...")
    # User context (constraints + intent)
    user_context = {
        "budget_norm": 0.6,
        "book_table": 1,
        "menu_diversity_norm": 0.7,
        "location": "banashankari",
        "budget_min": 500,
        "budget_max": 1200,
        "online_order": 1
    }
    
    # User preference (similarity)
    user_pref = {
        'log_cost': np.log(800),
        "menu_diversity": 5,
        "rest_type": ["casual dining", "buffet"],
        "cluster": 1,
        "is_specialized": 1
    }


    # Initialize pipeline
    pipeline = RecommendationPipeline(
        df=df,
        scenario_engine=ScenarioEngine(),
        candidate_generator=CandidateGenerator(df),
        filter_engine=HardFilter(df),
        ranker_class=Ranker,
        diversity_engine=AdaptiveDiversityReranker(),
        top_k_candidates=50
    )
    
    # Run recommendation
    output = pipeline.recommend(
        user_context=user_context,
        user_pref=user_pref,
        debug=True
    )
    
    # Inspect
    print("Detected Scenario:", output["scenario"])
    print("Logs:", output["logs"])
    
    print(
        output["recommendations"][
            ['name',"rest_type", "cluster", "votes",
             "final_score", "diversified_score"]
        ].head(5)
    )

Initializing engines...
Detected Scenario: date
Logs: {'scenario': {'scores': {'date': 0.45614035087719296, 'quick_bite': 0.14619883040935674, 'casual': 0.39766081871345027}, 'dominant_scenario': 'date'}, 'candidates_count': 50, 'filtered_count': 19, 'avg_final_score': np.float64(0.41033266890708514), 'unique_clusters': 1}
                         name      rest_type  cluster  votes  final_score  \
18893                  Haveli  casual dining        1     22     0.874440   
36522            Mayura Grand  casual dining        1     21     0.872853   
40461           Punjabi Tadka  casual dining        1    203     0.692361   
1767          New Kabab Plaza  casual dining        1      0     0.638566   
1953   Veruthe Oru Thattukada  casual dining        1    149     0.531384   

       diversified_score  
18893           0.961884  
36522           0.911150  
40461           0.700008  
1767            0.630744  
1953            0.515274  


In [23]:
df.to_parquet("restaurants_inference.parquet", index=False)

# streamlit_app.py

In [24]:
# from pipeline import RecommendationPipeline
# from scenario_engine import ScenarioEngine
# from candidate_generator import CandidateGenerator
# from hard_filter import HardFilter
# from ranker import Ranker
# from diversity_reranker import AdaptiveDiversityReranker

# ---------- Load Pipeline & Data once ----------
@st.cache_data
def load_pipeline():
    # Load only inference dataset (smaller parquet)
    df = pd.read_parquet("restaurants_inference.parquet")

    # Initialize layers
    scenario_engine = ScenarioEngine()
    candidate_generator = CandidateGenerator(df)
    filter_engine = HardFilter(df)
    ranker_class = Ranker
    diversity_engine = AdaptiveDiversityReranker()

    pipeline = RecommendationPipeline(
        df=df,
        scenario_engine=scenario_engine,
        candidate_generator=candidate_generator,
        filter_engine=filter_engine,
        ranker_class=ranker_class,
        diversity_engine=diversity_engine,
        top_k_candidates=50
    )
    return pipeline

pipeline = load_pipeline()

# ---------- Streamlit UI ----------
st.title("Restaurant Recommendation System")

# ---------- Sidebar Inputs ----------
st.sidebar.header("User Preferences")
budget = st.sidebar.slider("Budget (normalized 0-1)", 0.0, 1.0, 0.5)
book_table = st.sidebar.checkbox("Book Table?")
menu_diversity = st.sidebar.slider("Menu Diversity", 0, 5, 2)

rest_type = st.sidebar.multiselect(
    "Preferred Restaurant Type",
    ["casual dining", "buffet", "fine dining", "fast food"],
    default=["casual dining"]
)

st.sidebar.header("Hard Constraints")
location = st.sidebar.text_input("Location", "banashankari")
budget_min = st.sidebar.number_input("Budget Min", 0, 10000, 100)
budget_max = st.sidebar.number_input("Budget Max", 0, 10000, 1000)
online_order = st.sidebar.checkbox("Online Order Available?")

# ---------- Run Recommendation ----------
if st.button("Get Recommendations"):
    user_context = {
        "budget_norm": budget,
        "book_table": int(book_table),
        "menu_diversity_norm": menu_diversity,
        "location": location,
        "budget_min": budget_min,
        "budget_max": budget_max,
        "online_order": int(online_order)
    }

    user_pref = {
        "log_cost": np.log(800 * max(budget, 0.1)),
        "menu_diversity": menu_diversity,
        "rest_type": rest_type,
        "is_specialized": 0
    }

    # Run the pipeline
    result = pipeline.recommend(user_context, user_pref, debug=True)

    # ---------- Display Results ----------
    st.subheader("Top Recommendations")
    st.dataframe(
        result["recommendations"][["name", "rest_type", "final_score", "diversified_score", "cluster", "votes"]].head(10)
    )

    st.subheader("Logs")
    for k, v in result["logs"].items():
        st.write(f"{k}: {v}")

    st.write("Detected Scenario:", result["scenario"]["dominant_scenario"])

In [25]:
! streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py